In [1]:
from google.colab import files
uploaded = files.upload()

Saving DRiST_train_test_files.zip to DRiST_train_test_files.zip


In [3]:
import zipfile
with zipfile.ZipFile("DRiST_train_test_files.zip", "r") as z:
    z.extractall(".")

In [4]:
import pandas as pd

X_train_fe = pd.read_csv("X_train.csv")
X_test_fe = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").iloc[:, 0]
y_test = pd.read_csv("y_test.csv").iloc[:, 0]

print(X_train_fe.shape, X_test_fe.shape)

(55245, 31) (13812, 31)


In [5]:
from sklearn.feature_selection import mutual_info_classif

# ---- Method 1: correlation with target ----
train_with_target = X_train_fe.copy()
train_with_target["target"] = y_train.values
corr = train_with_target.corr(numeric_only=True)["target"].drop("target")
corr_rank = corr.abs().sort_values(ascending=False)

# ---- Method 2: mutual information (catches non-linear relationships correlation misses) ----
mi = mutual_info_classif(X_train_fe, y_train, random_state=42)
mi_series = pd.Series(mi, index=X_train_fe.columns).sort_values(ascending=False)

# ---- Combine both rankings ----
rank_df = pd.DataFrame({"correlation": corr_rank, "mutual_info": mi_series})
rank_df["corr_rank"] = rank_df["correlation"].rank(ascending=False)
rank_df["mi_rank"] = rank_df["mutual_info"].rank(ascending=False)
rank_df["avg_rank"] = (rank_df["corr_rank"] + rank_df["mi_rank"]) / 2
rank_df = rank_df.sort_values("avg_rank")

TOP_N = 15
selected_features = rank_df.head(TOP_N).index.tolist()

X_train_reduced = X_train_fe[selected_features]
X_test_reduced = X_test_fe[selected_features]

print(X_train_reduced.shape, X_test_reduced.shape)

(55245, 15) (13812, 15)


In [6]:
from sklearn.preprocessing import StandardScaler

scaler_reduced = StandardScaler()

X_train_final = pd.DataFrame(
    scaler_reduced.fit_transform(X_train_reduced),
    columns=X_train_reduced.columns,
    index=X_train_reduced.index
)
X_test_final = pd.DataFrame(
    scaler_reduced.transform(X_test_reduced),
    columns=X_test_reduced.columns,
    index=X_test_reduced.index
)

print(X_train_final.describe().T[["mean", "std"]].round(3))

                      mean  std
ComorbidityCount       0.0  1.0
BMI_Age_Interaction   -0.0  1.0
GenHlth               -0.0  1.0
HighBP                 0.0  1.0
PoorGenHlth            0.0  1.0
BMI                   -0.0  1.0
HighChol               0.0  1.0
Age                    0.0  1.0
DiffWalk              -0.0  1.0
BMI_Category_Obese    -0.0  1.0
BMI_Category_Normal   -0.0  1.0
Income                 0.0  1.0
PhysHlth              -0.0  1.0
HeartDiseaseorAttack  -0.0  1.0
UnhealthyDays          0.0  1.0
